In [9]:
import base64
from pwn import *
import rsa
from sympy.ntheory import factorint

In [10]:
HOST = '2f8d2590e81354948938de8b-1338-shadows-of-neon-city.challenge.cscg.live'

In [11]:
conn = connect(HOST, 1337, ssl=True)

[x] Opening connection to 2f8d2590e81354948938de8b-1338-shadows-of-neon-city.challenge.cscg.live on port 1337
[x] Opening connection to 2f8d2590e81354948938de8b-1338-shadows-of-neon-city.challenge.cscg.live on port 1337: Trying 147.28.207.143
[+] Opening connection to 2f8d2590e81354948938de8b-1338-shadows-of-neon-city.challenge.cscg.live on port 1337: Done


In [12]:
conn.recvline()
n_line = conn.recvline(keepends=False)
e_line = conn.recvline(keepends=False)
cipher_line = conn.recvline(keepends=False)
conn.recvline()

b'-----------------------------------------------------------\n'

In [13]:
n = int(n_line.split(b'= ')[-1].decode())
e = int(e_line.split(b'= ')[-1].decode())

primes = factorint(n)
p, q = primes.keys()

phi = (p - 1) * (q - 1)
d = pow(e, -1, phi)

pkey = rsa.PrivateKey(n=n, e=e, d=d, p=p, q=q)
pkey

PrivateKey(9196839056956368294087223928718119221, 65537, 8402428176058097068357969115394211073, 4954215442169478181, 1856366394298149841)

In [14]:
cipher = cipher_line.split(b'= ')[-1].decode()
cipher_int = int.from_bytes(base64.b64decode(cipher), 'big')
decrypt_int = pkey.blinded_decrypt(cipher_int)
pwd = decrypt_int.to_bytes((decrypt_int.bit_length() + 7) // 8, byteorder="big").decode()
print(pwd)

40d24a59d5e7


In [15]:
conn.recvline()
conn.recvline()
conn.send(pwd.encode())
conn.recvline()

b'V, inject this code into the mainframe backdoor: dach2025{curs3d_RSA_1s_curs3d_2124214}\n'